In [ ]:
from chi import server, context, lease, network
import chi, os, time, datetime

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")
username = os.getenv("USER")
print(username)

In [ ]:
l = lease.Lease(
    f"lease-ingest-{username}",
    duration=datetime.timedelta(hours=12)
)
l.add_flavor_reservation(
    id=chi.server.get_flavor_id("m1.xlarge"),
    amount=1
)
l.submit(idempotent=True)
l.show()

In [ ]:
import time

os_conn = chi.clients.connection()
cinder_client = chi.clients.cinder()

images_list = list(os_conn.image.images(name="CC-Ubuntu24.04"))
image_id = images_list[0].id
print("image_id:", image_id)

boot_vol = cinder_client.volumes.create(
    name=f"ingest-boot-{username}",
    size=150,
    imageRef=image_id,
)
print("volume id:", boot_vol.id)

while True:
    vol = cinder_client.volumes.get(boot_vol.id)
    print("status:", vol.status)
    if vol.status == "available":
        break
    time.sleep(10)

print("volume ready")

In [ ]:
bdm = [{
    "boot_index": 0,
    "uuid": boot_vol.id,
    "source_type": "volume",
    "destination_type": "volume",
    "delete_on_termination": True,
}]

srv = os_conn.compute.create_server(
    name=f"node-ingest-{username}",
    flavor_id=server.get_flavor_id(
        l.get_reserved_flavors()[0].name
    ),
    block_device_mapping_v2=bdm,
    networks=[{
        "uuid": os_conn.network.find_network("sharednet1").id
    }],
    key_name="id_rsa_chameleon",
)

print("waiting for ACTIVE...")
srv = os_conn.compute.wait_for_server(srv)
print("status:", srv.status)

In [ ]:
sharednet = os_conn.network.find_network("sharednet1")
port = next(
    p for p in os_conn.network.ports(device_id=srv.id)
    if p.network_id == sharednet.id
)
floating_net = os_conn.network.find_network("public")
fip = os_conn.network.create_ip(floating_network_id=floating_net.id)
os_conn.network.update_ip(fip, port_id=port.id)
print("floating IP:", fip.floating_ip_address)

security_groups = [
    {"name": "allow-ssh",  "port": 22,   "description": "SSH"},
    {"name": "allow-8888", "port": 8888, "description": "Jupyter"},
]

for sg in security_groups:
    secgroup = network.SecurityGroup({
        "name": sg["name"],
        "description": sg["description"],
    })
    secgroup.add_rule(direction="ingress", protocol="tcp", port=sg["port"])
    secgroup.submit(idempotent=True)
    os_conn.compute.add_security_group_to_server(srv, sg["name"])

print("security groups attached")

In [ ]:
s = server.get_server(f"node-ingest-{username}")
s.check_connectivity()

In [ ]:
context.choose_project()
context.choose_site(default="CHI@TACC")

conn = chi.clients.connection()
project_id = conn.current_project_id
identity_ep = conn.session.get_endpoint(service_type="identity", interface="public")
url = f"{identity_ep}/v3/users/{conn.current_user_id}/credentials/OS-EC2"

resp = conn.session.post(url, json={"tenant_id": project_id})
resp.raise_for_status()
ec2 = resp.json()["credential"]

print("EC2 Access:", ec2["access"])
print("EC2 Secret:", ec2["secret"])

In [ ]:

context.choose_site(default="KVM@TACC")
s_teardown = server.get_server(f"node-ingest-{username}")
s_teardown.delete()
print("VM deleted")